In [ ]:
# using h1 : counts how many tiles are out of place
from heapq import heappop, heappush

goal = (
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8)
)

start = (
    (0, 1, 3),
    (4, 2, 6),
    (7, 5, 8)
)

def misplaced_tiles(state):
    """Heuristic function h1: Counts how many tiles are out of place."""
    count = 0
    for i in range(3):
        for j in range(3):
            if state[i][j] != 0 and state[i][j] != goal[i][j]:
                count += 1
    return count

def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
            
def get_neighbors(state):
    x, y = find_blank(state)
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right
    neighbors = []
    for dx, dy in moves:
        nx = x + dx
        ny = y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            board = [list(row) for row in state]
            board[x][y], board[nx][ny] = board[nx][ny], board[x][y]
            neighbors.append(tuple(map(tuple, board)))
    return neighbors

def reconstruct(parent, current):
    """Traces the parent dictionary backward to build the solution path."""
    path = []
    while current is not None:
        path.append(current)
        current = parent[current]
    return path[::-1] # Reverse to get the path from start -> goal

def astar(start, heuristic):
    pq = []
    # Fixed: Pushing a tuple (f_cost, g_cost, state) instead of a set
    heappush(pq, (heuristic(start), 0, start))
    
    parent = {start: None}
    g_cost = {start: 0}
    expanded = 0
    
    while pq:
        f, g, current = heappop(pq)
        expanded += 1
        
        # Goal check
        if current == goal:
            return reconstruct(parent, current), expanded
        
        # Fixed: Loop through neighbors to process movements
        for neighbor in get_neighbors(current):
            tentative_g = g + 1
            
            # If neighbor hasn't been visited or a shorter path to it is found
            if neighbor not in g_cost or tentative_g < g_cost[neighbor]:
                g_cost[neighbor] = tentative_g
                f_cost = tentative_g + heuristic(neighbor)
                parent[neighbor] = current
                heappush(pq, (f_cost, tentative_g, neighbor))
                
    return None, expanded

# Execute the A* algorithm using h1 (misplaced_tiles)
path, nodes_expanded = astar(start, misplaced_tiles)

# Display final diagnostics and steps
print(f"Nodes Expanded: {nodes_expanded}")
print(f"Total Moves to Goal: {len(path) - 1}\n")
for step, state in enumerate(path):
    print(f"Step {step}:")
    for row in state:
        print(row)
    print()

Nodes Expanded: 8244
Total Moves to Goal: 22

Step 0:
(0, 1, 3)
(4, 2, 6)
(7, 5, 8)

Step 1:
(1, 0, 3)
(4, 2, 6)
(7, 5, 8)

Step 2:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 3:
(1, 2, 3)
(4, 6, 0)
(7, 5, 8)

Step 4:
(1, 2, 0)
(4, 6, 3)
(7, 5, 8)

Step 5:
(1, 0, 2)
(4, 6, 3)
(7, 5, 8)

Step 6:
(0, 1, 2)
(4, 6, 3)
(7, 5, 8)

Step 7:
(4, 1, 2)
(0, 6, 3)
(7, 5, 8)

Step 8:
(4, 1, 2)
(6, 0, 3)
(7, 5, 8)

Step 9:
(4, 1, 2)
(6, 3, 0)
(7, 5, 8)

Step 10:
(4, 1, 0)
(6, 3, 2)
(7, 5, 8)

Step 11:
(4, 0, 1)
(6, 3, 2)
(7, 5, 8)

Step 12:
(4, 3, 1)
(6, 0, 2)
(7, 5, 8)

Step 13:
(4, 3, 1)
(6, 5, 2)
(7, 0, 8)

Step 14:
(4, 3, 1)
(6, 5, 2)
(0, 7, 8)

Step 15:
(4, 3, 1)
(0, 5, 2)
(6, 7, 8)

Step 16:
(0, 3, 1)
(4, 5, 2)
(6, 7, 8)

Step 17:
(3, 0, 1)
(4, 5, 2)
(6, 7, 8)

Step 18:
(3, 1, 0)
(4, 5, 2)
(6, 7, 8)

Step 19:
(3, 1, 2)
(4, 5, 0)
(6, 7, 8)

Step 20:
(3, 1, 2)
(4, 0, 5)
(6, 7, 8)

Step 21:
(3, 1, 2)
(0, 4, 5)
(6, 7, 8)

Step 22:
(0, 1, 2)
(3, 4, 5)
(6, 7, 8)



In [7]:
print(get_neighbors(state=start))

[((4, 1, 3), (0, 2, 6), (7, 5, 8)), ((1, 0, 3), (4, 2, 6), (7, 5, 8))]


In [8]:
# using h2: calculates how far each tile is from its actual goal position.
def manhattan_distance(state):
    # Dictionary mapping each tile value to its exact row and column in the goal state
    goal_positions = {
        0: (0, 0), 1: (0, 1), 2: (0, 2),
        3: (1, 0), 4: (1, 1), 5: (1, 2),
        6: (2, 0), 7: (2, 1), 8: (2, 2)
    }
    
    total_distance = 0
    for i in range(3):
        for j in range(3):
            tile = state[i][j]
            # Skip the blank tile (0) per standard 8-puzzle rules
            if tile != 0:
                goal_i, goal_j = goal_positions[tile]
                # Formula: |x1 - x2| + |y1 - y2|
                total_distance += abs(i - goal_i) + abs(j - goal_j)
                
    return total_distance

path, nodes_expanded = astar(start, manhattan_distance)

# Output results
print(f"Nodes Expanded using h2: {nodes_expanded}")
print(f"Total Moves to Goal: {len(path) - 1}\n")
for step, state in enumerate(path):
    print(f"Step {step}:")
    for row in state:
        print(row)
    print()

Nodes Expanded using h2: 1109
Total Moves to Goal: 22

Step 0:
(0, 1, 3)
(4, 2, 6)
(7, 5, 8)

Step 1:
(1, 0, 3)
(4, 2, 6)
(7, 5, 8)

Step 2:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 3:
(1, 2, 3)
(4, 6, 0)
(7, 5, 8)

Step 4:
(1, 2, 0)
(4, 6, 3)
(7, 5, 8)

Step 5:
(1, 0, 2)
(4, 6, 3)
(7, 5, 8)

Step 6:
(0, 1, 2)
(4, 6, 3)
(7, 5, 8)

Step 7:
(4, 1, 2)
(0, 6, 3)
(7, 5, 8)

Step 8:
(4, 1, 2)
(6, 0, 3)
(7, 5, 8)

Step 9:
(4, 1, 2)
(6, 3, 0)
(7, 5, 8)

Step 10:
(4, 1, 0)
(6, 3, 2)
(7, 5, 8)

Step 11:
(4, 0, 1)
(6, 3, 2)
(7, 5, 8)

Step 12:
(4, 3, 1)
(6, 0, 2)
(7, 5, 8)

Step 13:
(4, 3, 1)
(6, 5, 2)
(7, 0, 8)

Step 14:
(4, 3, 1)
(6, 5, 2)
(0, 7, 8)

Step 15:
(4, 3, 1)
(0, 5, 2)
(6, 7, 8)

Step 16:
(0, 3, 1)
(4, 5, 2)
(6, 7, 8)

Step 17:
(3, 0, 1)
(4, 5, 2)
(6, 7, 8)

Step 18:
(3, 1, 0)
(4, 5, 2)
(6, 7, 8)

Step 19:
(3, 1, 2)
(4, 5, 0)
(6, 7, 8)

Step 20:
(3, 1, 2)
(4, 0, 5)
(6, 7, 8)

Step 21:
(3, 1, 2)
(0, 4, 5)
(6, 7, 8)

Step 22:
(0, 1, 2)
(3, 4, 5)
(6, 7, 8)



In [9]:
# using both h1 + h2
def combined_heuristic(state):
    return misplaced_tiles(state) + manhattan_distance(state)

path, nodes_expanded = astar(start, combined_heuristic)

# Output results
print(f"Nodes Expanded using (h1 + h2): {nodes_expanded}")
print(f"Total Moves to Goal: {len(path) - 1}\n")
for step, state in enumerate(path):
    print(f"Step {step}:")
    for row in state:
        print(row)
    print()

Nodes Expanded using (h1 + h2): 1115
Total Moves to Goal: 22

Step 0:
(0, 1, 3)
(4, 2, 6)
(7, 5, 8)

Step 1:
(1, 0, 3)
(4, 2, 6)
(7, 5, 8)

Step 2:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 3:
(1, 2, 3)
(4, 6, 0)
(7, 5, 8)

Step 4:
(1, 2, 0)
(4, 6, 3)
(7, 5, 8)

Step 5:
(1, 0, 2)
(4, 6, 3)
(7, 5, 8)

Step 6:
(0, 1, 2)
(4, 6, 3)
(7, 5, 8)

Step 7:
(4, 1, 2)
(0, 6, 3)
(7, 5, 8)

Step 8:
(4, 1, 2)
(6, 0, 3)
(7, 5, 8)

Step 9:
(4, 1, 2)
(6, 3, 0)
(7, 5, 8)

Step 10:
(4, 1, 0)
(6, 3, 2)
(7, 5, 8)

Step 11:
(4, 0, 1)
(6, 3, 2)
(7, 5, 8)

Step 12:
(4, 3, 1)
(6, 0, 2)
(7, 5, 8)

Step 13:
(4, 3, 1)
(6, 5, 2)
(7, 0, 8)

Step 14:
(4, 3, 1)
(6, 5, 2)
(0, 7, 8)

Step 15:
(4, 3, 1)
(0, 5, 2)
(6, 7, 8)

Step 16:
(0, 3, 1)
(4, 5, 2)
(6, 7, 8)

Step 17:
(3, 0, 1)
(4, 5, 2)
(6, 7, 8)

Step 18:
(3, 1, 0)
(4, 5, 2)
(6, 7, 8)

Step 19:
(3, 1, 2)
(4, 5, 0)
(6, 7, 8)

Step 20:
(3, 1, 2)
(4, 0, 5)
(6, 7, 8)

Step 21:
(3, 1, 2)
(0, 4, 5)
(6, 7, 8)

Step 22:
(0, 1, 2)
(3, 4, 5)
(6, 7, 8)

